In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.preprocessing.clean import preprocess_dataframe, create_splits

df = pd.read_csv("../data/raw/Bitext_support.csv")
print(f"Raw shape: {df.shape}")
df.head(3)

Raw shape: (26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...


In [2]:
# BERT version (for DistilBERT model)
df_bert = preprocess_dataframe(df, mode='bert')

# Classical version (for TF-IDF + Logistic Regression/SVM baseline)
df_classical = preprocess_dataframe(df, mode='classical')

# Compare before/after
for i in range(3):
    print(f"ORIGINAL : {df['instruction'].iloc[i]}")
    print(f"BERT     : {df_bert['text_clean'].iloc[i]}")
    print(f"CLASSICAL: {df_classical['text_clean'].iloc[i]}")
    print("---")

ORIGINAL : question about cancelling order {{Order Number}}
BERT     : question about cancelling order
CLASSICAL: question cancel order
---
ORIGINAL : i have a question about cancelling oorder {{Order Number}}
BERT     : i have a question about cancelling oorder
CLASSICAL: question cancel oorder
---
ORIGINAL : i need help cancelling puchase {{Order Number}}
BERT     : i need help cancelling puchase
CLASSICAL: need help cancel puchase
---


In [3]:
# Create splits for BERT version
train_bert, val_bert, test_bert = create_splits(df_bert)

print(f"Train: {len(train_bert)}")
print(f"Val:   {len(val_bert)}")
print(f"Test:  {len(test_bert)}")

# Verify stratification
print("\nCategory distribution in train:")
print(train_bert['label'].value_counts())

Train: 18810
Val:   4031
Test:  4031

Category distribution in train:
label
switch_account              700
contact_customer_service    700
edit_account                700
check_invoice               700
complaint                   700
registration_problems       699
cancel_order                699
newsletter_subscription     699
get_invoice                 699
contact_human_agent         699
track_refund                699
place_order                 699
delivery_period             699
check_payment_methods       699
payment_issue               699
get_refund                  698
review                      698
create_account              698
set_up_shipping_address     698
change_order                698
check_refund_policy         698
track_order                 697
delivery_options            697
delete_account              696
recover_password            696
change_shipping_address     681
check_cancellation_fee      665
Name: count, dtype: int64


In [4]:
train_bert.to_csv("../data/processed/train.csv", index=False)
val_bert.to_csv("../data/processed/val.csv", index=False)
test_bert.to_csv("../data/processed/test.csv", index=False)

# Also save classical version for baseline models
train_classical, val_classical, test_classical = create_splits(df_classical)
train_classical.to_csv("../data/processed/train_classical.csv", index=False)
val_classical.to_csv("../data/processed/val_classical.csv", index=False)
test_classical.to_csv("../data/processed/test_classical.csv", index=False)

print("All splits saved successfully")

All splits saved successfully


In [5]:
# Reload and verify
train = pd.read_csv("../data/processed/train.csv")
print(f"Train shape: {train.shape}")
print(f"Columns: {train.columns.tolist()}")
print(f"\nLabel distribution:\n{train['label'].value_counts()}")
print(f"\nSample row:")
print(train.iloc[0])

Train shape: (18810, 5)
Columns: ['text', 'text_clean', 'label', 'category', 'response']

Label distribution:
label
switch_account              700
contact_customer_service    700
edit_account                700
check_invoice               700
complaint                   700
registration_problems       699
cancel_order                699
newsletter_subscription     699
get_invoice                 699
contact_human_agent         699
track_refund                699
place_order                 699
delivery_period             699
check_payment_methods       699
payment_issue               699
get_refund                  698
review                      698
create_account              698
set_up_shipping_address     698
change_order                698
check_refund_policy         698
track_order                 697
delivery_options            697
delete_account              696
recover_password            696
change_shipping_address     681
check_cancellation_fee      665
Name: count, dtype: 